# Introduction NV Center (Work in Progress)
#### Author: Vineeth Thalakottoor, IE CNRS, LSDRM, CEA, Paris-Saclay
## Email: vineeth.thalakottoor@cea.fr or vineethfrancis.physics@gmail.com


In [1]:
# Define the source path
SourcePath = "/home/vineeth/Documents/PyOR/PyOR"

# Add source path
import sys
sys.path.append(SourcePath)

import time
%matplotlib ipympl

# Import PyOR package
from PyORv2 import *

*****        ***** *****
*   *        *   * *   *
*****  *   * *   * *****
*       * *  *   * * *  
*        *   ***** *  * 
        *               
       *                
Welcome to Python On Resonance (PyOR)

Author: Vineeth Thalakottoor, IE CNRS, LSDRM, CEA, Paris-Saclay

Email: vineeth.thalakottoor@cea.fr

"Everybody can simulate Magnetic Resonance"

Imported modules:

* QunS                   (QuantumSystem from PyOR_QuantumSystem)
** Hamiltonian           (from PyOR_Hamiltonian)
** DensityMatrix         (from PyOR_DensityMatrix)
** HardPulse             (from PyOR_HardPulse)
** Basis                 (from PyOR_Basis)
** RelaxationProcess     (from PyOR_Relaxation)
** Evolutions            (from PyOR_Evolution)
** Plotting              (from PyOR_Plotting)
** Spro                  (from PyOR_SignalProcessing)
** constants             (from PyOR_PhysicalConstants)
** gamma                 (from PyOR_Gamma)
* QunObj                 (from PyOR_QuantumObject)
* QuantumLibrary      

In [2]:
# Define the spin system
Spin_list = {"A" : "NV1", "B" : "C13"}
QS = QunS(Spin_list,PrintDefault=False)

Spin Operators generated (Quantum Objects)
For particle A: Ax, Ay, Az, Ap, Am
For particle B: Bx, By, Bz, Bp, Bm

To see the matrix form use the attribute .matrix


### Set parameters

In [ ]:
QS.Configure(
    # Propagation Space
    PropagationSpace="Hilbert",

    # Master Equation
    MasterEquation="Redfield",

    # Operator basis
    Basis_SpinOperators_Hilbert="Zeeman",

    # Field in TFalse
    B0=9.4,

    # Offset frequencies in Hz
    OFFSET={"A": 10.0, "B": 50.0},

    # J coupling
    Jcouplings=[("A", "B", 5.0)],

    # Initial and final spin temperature
    I_spintemp={"A": 300.0, "B": 300.0},
    F_spintemp={"A": 300.0, "B": 300.0},

    # Relaxation process
    Rprocess="Phenomenological",
    R1=1.0,
    R2=2.0,

    # Evolution parameters
    AcqDT = 0.0001,
    AcqAQ = 5.0,
    OdeMethod = 'DOP853',
    PropagationMethod = "ODE Solver",
    
    #Plotting
    PlotFigureSize = (10,5),
    PlotFontSize = 20
)

### Generate Hamiltonians

In [ ]:
Hz = QS.Hamiltonian.Zeeman_RotFrame()
Hz.Inverse2PI().Round(3).matrix

In [ ]:
# J coupling Hamiltonian
Hj = QS.Hamiltonian.Jcoupling()
Hj.Inverse2PI().Round(3).matrix

## Product Operator Basis (PMZ / Shift Z basis)

In [ ]:
sort = 'negative to positive'
Index = False
Normal = True
Basis_PMZ, coh_PMZ, dic_PMZ = QS.Basis.ProductOperators_SpinHalf_PMZ(sort,Index,Normal)

In [ ]:
Basis_PMZ[0].matrix

In [ ]:
print(type(Basis_PMZ))

### Product Operator Basis Zeeman

In [ ]:
Basis_Zeeman, dic_Zeeman, coh_Zeeman, coh_Zeeman_array = QS.Basis.ProductOperators_Zeeman()

## Initialize density matrix

In [ ]:
Thermal_DensMatrix = True

if Thermal_DensMatrix:    
    # High Temperature
    HT_approx = False
    
    # Initial Density Matrix
    rho_in = QS.DensityMatrix.EquilibriumDensityMatrix(QS.Ispintemp,HT_approx)
    
    # Equlibrium Density Matrix
    rhoeq = QS.DensityMatrix.EquilibriumDensityMatrix(QS.Fspintemp,HT_approx)
else:
    rho_in = QS.Az + QS.Bz
    rhoeq = QS.Az + QS.Bz

In [ ]:
QS.DensityMatrix.DensityMatrix_Components(rho_in,Basis_PMZ,dic_PMZ)

In [ ]:
QS.DensityMatrix.DensityMatrix_Components(rho_in,Basis_Zeeman, dic_Zeeman)

In [ ]:
# Initial Density Matrix
rho_in.matrix

In [ ]:
# Final Density Matrix
rhoeq.matrix

## Hard Pulse

In [ ]:
flip_angle1 = 90.0   # Flip angle Spin 1
flip_angle2 = 90.0 # Flip angle Spin 2

rho = QS.HardPulse.Rotate_Pulse(rho_in,flip_angle1,QS.Ay)
rho = QS.HardPulse.Rotate_Pulse(rho,flip_angle2,QS.By) 

In [ ]:
rho.matrix

In [ ]:
rho.matrix

In [ ]:
QS.DensityMatrix.DensityMatrix_Components(rho,Basis_PMZ,dic_PMZ)

In [ ]:
QS.DensityMatrix.DensityMatrix_Components(rho,Basis_Zeeman, dic_Zeeman)

## Evolution

In [ ]:
start_time = time.time()
t, rho_t = QS.Evolutions.Evolution(rho,rhoeq,Hz+Hj)
end_time = time.time()
timetaken = end_time - start_time
print("Total time = %s seconds " % (timetaken))

## Expectation

In [ ]:
det_Mt = QS.Ap + QS.Bp
det_Z = QS.Az + QS.Bz

t, Mt = QS.Evolutions.Expectation(rho_t,det_Mt)
t, Mz = QS.Evolutions.Expectation(rho_t,det_Z)

## Plotting

In [ ]:
QS.Plotting.Plotting_SpanSelector(t,Mt,"time (s)","Mt","red") 

In [ ]:
QS.Plotting.Plotting_SpanSelector(t,Mz,"time (s)","Mz","red") 

## Fourier Transform

In [ ]:
freq, spectrum = QS.Spro.FourierTransform(Mt,QS.AcqFS,5)

In [ ]:
QS.Plotting.PlotXlimt= (-60,0)
#QS.Plotting.PlotYlimt= (0,3500)
QS.Plotting.Plotting_SpanSelector(freq,spectrum,"Frequency (Hz)","Spectrum","red")